# TRELLIS.2 geometry LoRA fine-tuning (footwear) - Colab, end to end

Implements the Hugging Face blog post *The Definitive Fine-Tuning Guide for 3D Generation Models*
(https://huggingface.co/blog/3d-lora-finetuning) using its companion code
(https://github.com/suvadityamuk/trellis2-finetuning), without Docker / HF Jobs.

Pipeline: build env -> (optional) preprocess your own meshes -> LoRA-train the sparse-structure flow
-> LoRA-train the shape flow -> evaluate base vs LoRA on held-out objects -> generate a textured GLB.

**Before running**

1. Runtime -> Change runtime type -> **L4 or A100** (>= 24 GB VRAM, High-RAM). T4 is not supported (16 GB, no flash-attn target).
2. Accept the DINOv3 license with your HF account: https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m
3. Add an HF token as the Colab secret `HF_TOKEN` (read access is enough; write only if you set `PUSH_TO`).

**Budget** (blog numbers, 3000 steps, A100 80 GB): sparse-structure flow 7.2 h, shape flow 3.6 h. An L4 is slower.
Checkpoints are written every `I_SAVE` steps and every training cell resumes from the newest one, so a
disconnected session just needs "Run all" again (keep `USE_DRIVE = True`).

## 0. Config

In [ ]:
# ---- what to run -------------------------------------------------------------------------
TRAIN_SS               = True     # LoRA on the sparse-structure flow (rank 8, attention + MLP)
TRAIN_SHAPE            = True     # LoRA on the shape flow (rank 16, attention only)
USE_PUBLISHED_ADAPTERS = False    # True: skip training, evaluate / infer with the blog's adapters
RUN_EVAL               = True     # base vs LoRA on the held-out objects (Chamfer, F-score, normals)
RUN_INFERENCE          = True     # image -> textured GLB, base and LoRA

# ---- data --------------------------------------------------------------------------------
LATENTS        = "suvadityamuk/gso-footwear-trellis2-latents"   # the blog's preprocessed 188 GSO shoes
USE_OWN_MESHES = False            # True: run the 8-stage preprocessing on MESHES instead of LATENTS
MESHES         = "/content/my_meshes"   # local folder or Hub dataset repo id (.glb/.obj/.fbx/.ply/.stl/...)
HOLDOUT_FRAC   = 0.05

# ---- training (blog values: 3000 steps, save every 500, effective batch 16, lr 1e-4) -----
MAX_STEPS  = 1000                 # set 3000 to reproduce the blog exactly
I_SAVE     = 250
BATCH_SIZE = None                 # None = auto from VRAM (4 on 80 GB, else 2)
GRAD_ACCUM = None                 # None = auto so that BATCH_SIZE * GRAD_ACCUM = 16
LR         = 1e-4

# ---- outputs -----------------------------------------------------------------------------
USE_DRIVE     = True              # keep adapters + compiled wheels on Google Drive (resume / fast rebuild)
PUSH_TO       = ""                # optional model repo "user/name": every checkpoint is uploaded
TRACKIO_SPACE = ""                # optional Space "user/name" for a hosted loss dashboard

# ---- evaluation / inference --------------------------------------------------------------
PIPELINE_TYPE = "512"             # "512" fits 24 GB; "1024_cascade" needs more
EVAL_MAX_OBJECTS = 0              # 0 = all held-out objects
INFER_IMAGE   = ""                # path to your own shoe image; empty = first held-out render
SEED          = 42

## 1. Runtime checks, paths, token

In [ ]:
import os, re, json, glob, shlex, subprocess, sys

def sh(cmd, log=None, cwd=None, check=True):
    # Run a shell command, stream its output, optionally append it to a log file.
    print("+", cmd, flush=True)
    p = subprocess.Popen(["bash", "-o", "pipefail", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ.copy())
    f = open(log, "a") if log else None
    for line in p.stdout:
        print(line, end="", flush=True)
        if f:
            f.write(line); f.flush()
    if f:
        f.close()
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc

# ---- GPU -----------------------------------------------------------------------------------
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                   capture_output=True, text=True)
assert q.returncode == 0, "No GPU. Runtime -> Change runtime type -> L4 or A100."
GPU_NAME, vram_mib = [s.strip() for s in q.stdout.strip().splitlines()[0].split(",")]
VRAM_GB = int(vram_mib) / 1024
print(f"GPU: {GPU_NAME}  {VRAM_GB:.0f} GB")
assert VRAM_GB >= 21, f"{GPU_NAME} has {VRAM_GB:.0f} GB; TRELLIS.2 LoRA training needs >= 24 GB (L4 / A100)."

if BATCH_SIZE is None:
    BATCH_SIZE = 4 if VRAM_GB >= 70 else 2
if GRAD_ACCUM is None:
    GRAD_ACCUM = max(1, 16 // BATCH_SIZE)
print(f"batch {BATCH_SIZE} x accum {GRAD_ACCUM} = effective {BATCH_SIZE * GRAD_ACCUM}")

# ---- paths ---------------------------------------------------------------------------------
ROOT    = "/content"
REL     = f"{ROOT}/trellis2-finetuning"      # companion repo (preprocess.py, finetune.py, patch)
TRELLIS = f"{ROOT}/TRELLIS.2"                # patched upstream checkout
WORK    = f"{ROOT}/work"
DATA    = f"{WORK}/data"                     # latents + renders (local disk: fast IO)
PERSIST = WORK
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/trellis2_lora"
OUT     = f"{PERSIST}/results"               # <OUT>/ss_lora, <OUT>/shape_lora
WHEELS  = f"{PERSIST}/wheels"                # compiled CUDA extensions, reused across sessions
EVAL    = f"{WORK}/eval"
GEN     = f"{WORK}/generated"
for d in (WORK, DATA, OUT, WHEELS, EVAL, GEN):
    os.makedirs(d, exist_ok=True)

# ---- token ---------------------------------------------------------------------------------
token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
if not token:
    from getpass import getpass
    token = getpass("HF token: ").strip()
os.environ["HF_TOKEN"] = token
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRELLIS2_ROOT"] = TRELLIS
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
print("paths ok; results ->", OUT)

## 2. Build the environment

Mirrors the blog's `container/Dockerfile`: upstream TRELLIS.2 at commit `75fbf01` + the release patch,
torch 2.6.0 / cu124, transformers 5.18.0, flash-attn 2.7.3, and the compiled extensions
(nvdiffrast, nvdiffrec, CuMesh, FlexGEMM, o-voxel).

First run compiles for ~20-40 min; the wheels are cached in `WHEELS`, so later sessions take a few minutes.
torch is only ever imported in subprocesses, so no runtime restart is needed.

In [ ]:
SETUP = r'''
set -euo pipefail
REL="$1"; TRELLIS="$2"; WHEEL_ROOT="$3"
REL_COMMIT=d2a6a81b88ee5dfbcbbb5594be7fdda95c0a8922
TRELLIS_COMMIT=75fbf0183001ed9876c8dbb35de6b68552ee08bd
export DEBIAN_FRONTEND=noninteractive
export CUDA_HOME=/usr/local/cuda
export PATH="$CUDA_HOME/bin:$PATH"
command -v nvcc >/dev/null || { echo "nvcc not found under $CUDA_HOME: the CUDA toolkit is required to compile the extensions"; exit 1; }

echo "== system packages"
apt-get -qq update >/dev/null
apt-get -qq install -y xvfb libgl1 libegl1 libglu1-mesa libxrender1 libxi6 libxkbcommon-x11-0 libsm6 \
    libxfixes3 libxrandr2 libxxf86vm1 libjpeg-dev >/dev/null

echo "== sources"
if [ ! -d "$REL/.git" ]; then git clone -q https://github.com/suvadityamuk/trellis2-finetuning "$REL"; fi
git -C "$REL" checkout -q "$REL_COMMIT"
if [ ! -f "$TRELLIS/.patched" ]; then
    rm -rf "$TRELLIS"
    git clone -q https://github.com/microsoft/TRELLIS.2.git "$TRELLIS"
    git -C "$TRELLIS" checkout -q "$TRELLIS_COMMIT"
    git -C "$TRELLIS" apply "$REL/container/patches/trellis2-75fbf01.patch"
    cp -r "$REL/container/trellis2/." "$TRELLIS/"          # LoRA configs, trainer, eval script, mesh adapter
    touch "$TRELLIS/.patched"
fi

echo "== python packages"
CONS=/content/constraints.txt
printf "torch==2.6.0\ntorchvision==0.21.0\ntransformers==5.18.0\n" > "$CONS"
pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
pip install -q -c "$CONS" imageio imageio-ffmpeg tqdm easydict ninja trimesh transformers==5.18.0 tensorboard \
    pandas lpips zstandard kornia timm plyfile scipy opencv-python-headless \
    peft accelerate trackio safetensors huggingface_hub
pip install -q -c "$CONS" git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8
python - <<'PY'
import torch
assert torch.__version__.startswith("2.6.0"), f"torch got replaced: {torch.__version__}"
assert torch.cuda.is_available(), "torch cannot see the GPU"
print("torch", torch.__version__, "| cuda", torch.version.cuda, "|", torch.cuda.get_device_name(0))
PY

PY=$(python -c 'import sys; print(f"cp{sys.version_info[0]}{sys.version_info[1]}")')
ABI=$(python -c 'import torch; print(str(torch._C._GLIBCXX_USE_CXX11_ABI).upper())')
ARCH=$(python -c 'import torch; print("%d.%d" % torch.cuda.get_device_capability())')
export TORCH_CUDA_ARCH_LIST="$ARCH"          # compile kernels only for this GPU family
export MAX_JOBS=$(python -c 'import os; print(max(1, min(4, os.cpu_count() or 1)))')
W="$WHEEL_ROOT/${PY}-torch2.6.0cu124-sm${ARCH}"
mkdir -p "$W"
echo "== extensions (python $PY, sm $ARCH, MAX_JOBS=$MAX_JOBS, wheel cache $W)"

echo "-- flash-attn"
python -c "import flash_attn" 2>/dev/null || \
pip install -q "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.3/flash_attn-2.7.3+cu12torch2.6cxx11abi${ABI}-${PY}-${PY}-linux_x86_64.whl" \
    || pip install -q flash-attn==2.7.3 --no-build-isolation

SRC=/tmp/extensions; mkdir -p "$SRC"
ext() {   # ext <key> <import name> <command that fetches the source into $SRC/<key>>
    local key="$1" mod="$2"; shift 2
    if python -c "import $mod" 2>/dev/null; then echo "-- $key: already installed"; return; fi
    if ! ls "$W/$key"/*.whl >/dev/null 2>&1; then
        echo "-- $key: building"
        rm -rf "$SRC/$key" "$W/$key"; "$@"
        pip wheel -q --no-deps --no-build-isolation -w "$W/$key" "$SRC/$key"
    else
        echo "-- $key: cached wheel"
    fi
    pip install -q --no-deps --force-reinstall "$W/$key"/*.whl
}
fetch_ovoxel() {   # the eigen submodule is not checked out by a plain clone
    cp -r "$TRELLIS/o-voxel" "$SRC/o_voxel"
    rm -rf "$SRC/o_voxel/third_party/eigen"
    git clone -q --depth 1 -b 3.4.0 https://gitlab.com/libeigen/eigen.git "$SRC/o_voxel/third_party/eigen"
}
ext nvdiffrast nvdiffrast git clone -q -b v0.4.0 https://github.com/NVlabs/nvdiffrast.git "$SRC/nvdiffrast"
ext nvdiffrec  nvdiffrec_render git clone -q -b renderutils https://github.com/JeffreyXiang/nvdiffrec.git "$SRC/nvdiffrec"
ext cumesh     cumesh    git clone -q --recursive https://github.com/JeffreyXiang/CuMesh.git "$SRC/cumesh"
ext flex_gemm  flex_gemm git clone -q --recursive https://github.com/JeffreyXiang/FlexGEMM.git "$SRC/flex_gemm"
ext o_voxel    o_voxel   fetch_ovoxel

echo "== import check"
cd "$TRELLIS"
python - <<'PY'
import torch, flash_attn, flex_gemm, cumesh, o_voxel, nvdiffrast.torch, peft, accelerate, transformers
from trellis2 import models, datasets
from trellis2.pipelines import Trellis2ImageTo3DPipeline
print("transformers", transformers.__version__, "| peft", peft.__version__, "| flash_attn", flash_attn.__version__)
print("ENV OK")
PY
'''
open(f"{ROOT}/setup_env.sh", "w").write(SETUP)
sh(f"bash {ROOT}/setup_env.sh {REL} {TRELLIS} {WHEELS}", log=f"{WORK}/setup.log")

## 3. Hub access check (DINOv3 is gated)

In [ ]:
CHECK = r'''
import os, sys
from huggingface_hub import HfApi, hf_hub_download
api = HfApi()
print("logged in as:", api.whoami()["name"])
try:
    hf_hub_download("facebook/dinov3-vitl16-pretrain-lvd1689m", "config.json")
    print("DINOv3 access: ok")
except Exception as e:
    sys.exit("DINOv3 access DENIED. Accept the license at "
             "https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m with this token's account, "
             f"then re-run.\n{type(e).__name__}: {e}")
push_to = os.environ.get("PUSH_TO_REPO", "")
if push_to:
    api.create_repo(push_to, exist_ok=True)
    print("push repo ok:", push_to)
'''
open(f"{ROOT}/check_hub.py", "w").write(CHECK)
os.environ["PUSH_TO_REPO"] = PUSH_TO
sh(f"python {ROOT}/check_hub.py")

## 4. Data

Default: download the blog's preprocessed dataset (188 Google Scanned Objects shoes: sparse-structure latents,
512 shape latents, 16 conditioning renders per object, held-out split, ground-truth meshes; ~1.9 GB).

`USE_OWN_MESHES = True` instead runs the blog's 8-stage preprocessing (Blender mesh dump -> O-Voxel dual grids ->
shape latents -> sparse-structure latents -> 16 Cycles renders -> held-out split -> GLB export) on your meshes.
It resumes if re-run. The blog measured ~1 h for 188 objects on an A100.

In [ ]:
if USE_OWN_MESHES:
    BLENDER = r'''
set -euo pipefail
cd /tmp
get() {   # download.blender.org rejects datacenter IPs; use the mirror the blog's image uses
    local dir="$1" name="$2"
    [ -x "/tmp/$name/blender" ] && return
    wget -q --tries=3 --waitretry=10 -U "Mozilla/5.0" "https://ftp.halifax.rwth-aachen.de/blender/release/$dir/$name.tar.xz" -P /tmp
    tar -xf "/tmp/$name.tar.xz" -C /tmp && rm "/tmp/$name.tar.xz"
}
get Blender3.0 blender-3.0.1-linux-x64      # render_cond.py (Cycles conditioning views)
get Blender4.5 blender-4.5.1-linux-x64      # dump_mesh.py
echo "blender ok"
'''
    open(f"{ROOT}/get_blender.sh", "w").write(BLENDER)
    sh(f"bash {ROOT}/get_blender.sh")
    workers = max(1, min(8, (os.cpu_count() or 2) - 1))
    sh(f"python preprocess.py --meshes {shlex.quote(MESHES)} --out {DATA} --trellis-root {TRELLIS} "
       f"--holdout {HOLDOUT_FRAC} --views 16 --workers {workers}", cwd=REL, log=f"{WORK}/preprocess.log")
else:
    FETCH = r'''
import sys
from huggingface_hub import snapshot_download
repo, dst = sys.argv[1], sys.argv[2]
snapshot_download(repo, repo_type="dataset", local_dir=dst, allow_patterns=[
    "metadata.csv", "holdout.txt", "eval_testlist.txt",
    "ss_latents/ss_enc_conv3d_16l8_fp16_64/*",
    "shape_latents/shape_enc_next_dc_f16c32_fp16_512/*",
    "renders_cond/**", "holdout_gt/*"])
print("downloaded", repo, "->", dst)
'''
    open(f"{ROOT}/fetch_latents.py", "w").write(FETCH)
    sh(f"python {ROOT}/fetch_latents.py {shlex.quote(LATENTS)} {DATA}")

# ---- sanity + a local eval list (the published one holds paths from the machine that made it) ----
n_ss    = len(glob.glob(f"{DATA}/ss_latents/ss_enc_conv3d_16l8_fp16_64/*.npz"))
n_shape = len(glob.glob(f"{DATA}/shape_latents/shape_enc_next_dc_f16c32_fp16_512/*.npz"))
n_rend  = len(glob.glob(f"{DATA}/renders_cond/*/transforms.json"))
holdout = open(f"{DATA}/holdout.txt").read().split() if os.path.exists(f"{DATA}/holdout.txt") else []
print(f"ss latents {n_ss} | shape latents {n_shape} | rendered objects {n_rend} | held out {len(holdout)}")
assert n_ss and n_shape and n_rend, "dataset is incomplete - check the download / preprocessing log above"

TESTLIST = f"{WORK}/eval_testlist_local.txt"
lines = []
for sha in holdout:
    tf = f"{DATA}/renders_cond/{sha}/transforms.json"
    if os.path.exists(tf) and os.path.exists(f"{DATA}/holdout_gt/{sha}.glb"):
        frame = json.load(open(tf))["frames"][0]["file_path"]
        lines.append(f"{sha} {os.path.join(DATA, 'renders_cond', sha, frame)}")
open(TESTLIST, "w").write("\n".join(lines) + "\n")
print(f"eval list: {len(lines)} objects -> {TESTLIST}")

## 5. Train the LoRA adapters

`finetune.py` from the blog's repo, unchanged. Per flow it:

- loads the frozen 1.3B flow DiT from `microsoft/TRELLIS.2-4B`, enables gradient checkpointing on the blocks
- injects PEFT LoRA with `inject_adapter_in_model` (keeps the custom forward): **ss** rank 8 / alpha 16 on
  `to_qkv, to_out, to_q, to_kv, mlp.mlp.0, mlp.mlp.2`; **shape** rank 16 / alpha 32 on attention only
  (its MLP is `SparseLinear`, which PEFT cannot wrap)
- trains only the LoRA weights (fp32) under bf16 autocast with the rectified-flow velocity loss,
  `x_t = (1-t) x0 + (sigma_min + (1-sigma_min) t) eps`, target `(1-sigma_min) eps - x0`, timestep x1000;
  t is logit-normal(1, 1) for ss and uniform for shape; DINOv3-L conditioning with 10% CFG dropout
- AdamW lr 1e-4, betas (0.9, 0.95), wd 0.01, grad-clip 1.0; held-out objects are excluded

Re-running a cell resumes from the newest `adapter-<step>.safetensors` in `OUT` (or `PUSH_TO`).

In [ ]:
def train(flow):
    run_dir = f"{OUT}/{flow}_lora"
    os.makedirs(run_dir, exist_ok=True)
    workers = max(1, min(4, (os.cpu_count() or 2) - 1))
    cmd = (f"python finetune.py --flow {flow} --latents {DATA} --output {OUT} --trellis-root {TRELLIS} "
           f"--set max_steps={MAX_STEPS} i_save={I_SAVE} batch_size={BATCH_SIZE} grad_accum={GRAD_ACCUM} "
           f"lr={LR} num_workers={workers}")
    if PUSH_TO:
        cmd += f" --push-to {shlex.quote(PUSH_TO)}"
    cmd += f" --trackio-space {shlex.quote(TRACKIO_SPACE)}" if TRACKIO_SPACE else " --no-trackio"
    sh(cmd, cwd=REL, log=f"{run_dir}/train.log")

if TRAIN_SS and not USE_PUBLISHED_ADAPTERS:
    train("ss")

In [ ]:
if TRAIN_SHAPE and not USE_PUBLISHED_ADAPTERS:
    train("shape")

### Loss curves

In [ ]:
import matplotlib.pyplot as plt

pat = re.compile(r"step (\d+)/\d+\s+loss ([\d.eE+-]+|nan|inf)\s+ema ([\d.eE+-]+|nan|inf)")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, flow in zip(axes, ("ss", "shape")):
    log = f"{OUT}/{flow}_lora/train.log"
    pts = {}
    if os.path.exists(log):
        for m in pat.finditer(open(log, errors="ignore").read()):
            pts[int(m.group(1))] = (float(m.group(2)), float(m.group(3)))
    if pts:
        steps = sorted(pts)
        ax.plot(steps, [pts[s][0] for s in steps], alpha=0.35, label="loss")
        ax.plot(steps, [pts[s][1] for s in steps], label="EMA")
        ax.legend()
    ax.set_title(f"{flow} flow LoRA" + ("" if pts else " (no log)"))
    ax.set_xlabel("optimizer step"); ax.set_ylabel("velocity MSE")
plt.tight_layout(); plt.show()

## 6. Locate the adapters

In [ ]:
def latest_adapter(run_dir):
    best = (0, None)
    for p in glob.glob(f"{run_dir}/adapter-*.safetensors"):
        m = re.search(r"adapter-(\d+)\.safetensors$", p)
        if m and int(m.group(1)) >= best[0]:
            best = (int(m.group(1)), p)
    return best[1]

SS_CONFIG    = f"{TRELLIS}/configs/lora/ss_flow_lora_gso.json"
SHAPE_CONFIG = f"{TRELLIS}/configs/lora/slat_flow_img2shape_lora_gso.json"

if USE_PUBLISHED_ADAPTERS:
    PULL = r'''
import re, sys, json
from huggingface_hub import HfApi, hf_hub_download
repo = "suvadityamuk/trellis-2-finetune-gso"
best = {}
for f in HfApi().list_repo_files(repo):
    m = re.search(r"adapter-(\d+)\.safetensors$", f)
    if m:
        flow = "shape" if "shape" in f.lower() or "slat" in f.lower() else "ss"
        if int(m.group(1)) >= best.get(flow, (-1, None))[0]:
            best[flow] = (int(m.group(1)), f)
if set(best) != {"ss", "shape"}:
    sys.exit(f"could not find both adapters in {repo}; files matched: {best}")
out = {k: hf_hub_download(repo, v[1]) for k, v in best.items()}
print({k: v[1] for k, v in best.items()})
json.dump(out, open(sys.argv[1], "w"))
'''
    open(f"{ROOT}/pull_adapters.py", "w").write(PULL)
    sh(f"python {ROOT}/pull_adapters.py {WORK}/published_adapters.json")
    pub = json.load(open(f"{WORK}/published_adapters.json"))
    SS_ADAPTER, SHAPE_ADAPTER = pub["ss"], pub["shape"]
else:
    SS_ADAPTER, SHAPE_ADAPTER = latest_adapter(f"{OUT}/ss_lora"), latest_adapter(f"{OUT}/shape_lora")
    # rank / alpha / targets come from the config each run saved next to its adapters
    if os.path.exists(f"{OUT}/ss_lora/config.json"):
        SS_CONFIG = f"{OUT}/ss_lora/config.json"
    if os.path.exists(f"{OUT}/shape_lora/config.json"):
        SHAPE_CONFIG = f"{OUT}/shape_lora/config.json"

print("ss adapter   :", SS_ADAPTER)
print("shape adapter:", SHAPE_ADAPTER)
assert SS_ADAPTER or SHAPE_ADAPTER, "no adapter found - train first or set USE_PUBLISHED_ADAPTERS = True"
ADAPTER_ARGS = ""
if SS_ADAPTER:
    ADAPTER_ARGS += f" --ss_adapter {shlex.quote(SS_ADAPTER)} --ss_config {shlex.quote(SS_CONFIG)}"
if SHAPE_ADAPTER:
    ADAPTER_ARGS += f" --shape_adapter {shlex.quote(SHAPE_ADAPTER)} --shape_config {shlex.quote(SHAPE_CONFIG)}"

## 7. Evaluate: base vs LoRA on the held-out objects

The blog's `scripts/eval_geometry.py`: one conditioning render per held-out object, a mesh from the base
pipeline and from the LoRA pipeline with the same seed, each compared to the ground-truth mesh after
normalisation. Blog result at 3000 steps (9 shoes): F-score@0.02 0.276 -> 0.803, Chamfer 0.174 -> 0.033,
normal consistency 0.530 -> 0.804.

In [ ]:
if RUN_EVAL and not open(TESTLIST).read().strip():
    print("no held-out objects with ground truth - skipping evaluation")
elif RUN_EVAL:
    cmd = (f"python scripts/eval_geometry.py --test_list {TESTLIST} --mesh_dir {DATA}/holdout_gt"
           f"{ADAPTER_ARGS} --out_dir {EVAL} --pipeline_type {PIPELINE_TYPE} --tau 0.02 --num_points 50000"
           f" --seed {SEED} --max_objects {EVAL_MAX_OBJECTS}")
    sh(cmd, cwd=TRELLIS, log=f"{WORK}/eval.log")

    import pandas as pd
    df = pd.read_csv(f"{EVAL}/metrics.csv")
    summary = (df[df.object_id == "__mean__"].set_index("model")
               [["fscore", "chamfer", "normal_consistency", "precision", "recall"]])
    display(summary.round(4))
    per_obj = df[df.object_id != "__mean__"].pivot(index="object_id", columns="model",
                                                    values=["fscore", "chamfer"]).round(4)
    display(per_obj)
    sh(f"mkdir -p {PERSIST}/eval && cp {EVAL}/metrics.csv {PERSIST}/eval/metrics.csv", check=False)

## 8. Inference: image -> textured GLB (base and LoRA)

Loads `Trellis2ImageTo3DPipeline`, generates with the base weights, injects both adapters into
`sparse_structure_flow_model` and the shape flow(s), generates again with the same seed, and exports
both as textured GLBs through `o_voxel.postprocess.to_glb`. The texture flow is untouched, as in the blog.

In [ ]:
INFER = r'''
import os, sys, argparse
os.environ.setdefault("OPENCV_IO_ENABLE_OPENEXR", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
ap = argparse.ArgumentParser()
ap.add_argument("--trellis_root", required=True)
ap.add_argument("--image", required=True)
ap.add_argument("--out_dir", required=True)
ap.add_argument("--ss_adapter"); ap.add_argument("--ss_config")
ap.add_argument("--shape_adapter"); ap.add_argument("--shape_config")
ap.add_argument("--pipeline_type", default="512")
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--decimation", type=int, default=200000)
ap.add_argument("--texture_size", type=int, default=2048)
args = ap.parse_args()
args.image, args.out_dir = os.path.abspath(args.image), os.path.abspath(args.out_dir)
sys.path.insert(0, args.trellis_root)
os.chdir(args.trellis_root)

import torch
from PIL import Image
import o_voxel
from trellis2.pipelines import Trellis2ImageTo3DPipeline
from scripts.eval_geometry import apply_adapter, load_lora_spec, shape_keys, SS_KEY, SS_TARGETS, SHAPE_TARGETS

torch.set_grad_enabled(False)
os.makedirs(args.out_dir, exist_ok=True)
pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()
image = Image.open(args.image)

def generate(tag):
    mesh = pipeline.run(image, num_samples=1, seed=args.seed, pipeline_type=args.pipeline_type)[0]
    mesh.simplify(16777216)                                # nvdiffrast limit
    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs, coords=mesh.coords,
        attr_layout=mesh.layout, voxel_size=mesh.voxel_size,
        aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target=args.decimation, texture_size=args.texture_size,
        remesh=True, remesh_band=1, remesh_project=0, verbose=False)
    path = os.path.join(args.out_dir, f"{tag}.glb")
    glb.export(path, extension_webp=True)
    print(f"[{tag}] {mesh.vertices.shape[0]} verts -> {path}", flush=True)
    del mesh, glb
    torch.cuda.empty_cache()

generate("base")
if args.ss_adapter:
    r, a, t = load_lora_spec(args.ss_config, 8, 16, SS_TARGETS)
    apply_adapter(pipeline.models[SS_KEY], args.ss_adapter, r, a, t, merge=False)
if args.shape_adapter:
    r, a, t = load_lora_spec(args.shape_config, 16, 32, SHAPE_TARGETS)
    for key in shape_keys(args.pipeline_type):
        apply_adapter(pipeline.models[key], args.shape_adapter, r, a, t, merge=False)
generate("lora")
print("INFERENCE DONE")
'''
open(f"{ROOT}/infer_lora.py", "w").write(INFER)

if RUN_INFERENCE:
    image = INFER_IMAGE
    if not image:                                         # first held-out render, else any render
        first = open(TESTLIST).read().strip().split("\n")[0]
        if first:
            image = first.split(" ", 1)[1]
        else:
            tf = sorted(glob.glob(f"{DATA}/renders_cond/*/transforms.json"))[0]
            image = os.path.join(os.path.dirname(tf), json.load(open(tf))["frames"][0]["file_path"])
    assert os.path.exists(image), f"image not found: {image}"
    print("input image:", image)
    sh(f"python {ROOT}/infer_lora.py --trellis_root {TRELLIS} --image {shlex.quote(image)} --out_dir {GEN}"
       f"{ADAPTER_ARGS} --pipeline_type {PIPELINE_TYPE} --seed {SEED}", log=f"{WORK}/infer.log")
    sh(f"mkdir -p {PERSIST}/generated && cp {GEN}/*.glb {PERSIST}/generated/", check=False)

### View the result

In [ ]:
import base64
from IPython.display import HTML, Image as IPyImage, display

def glb_viewer(paths, height=420):
    blocks = []
    for label, path in paths:
        size_mb = os.path.getsize(path) / 1e6
        if size_mb > 40:
            blocks.append(f"<div style='flex:1'><b>{label}</b><br>{size_mb:.0f} MB - too large to preview; download {path}</div>")
            continue
        b64 = base64.b64encode(open(path, "rb").read()).decode()
        blocks.append(f"<div style='flex:1'><b>{label}</b> ({size_mb:.1f} MB)"
                      f"<model-viewer src='data:model/gltf-binary;base64,{b64}' camera-controls auto-rotate "
                      f"style='width:100%;height:{height}px;background:#eee'></model-viewer></div>")
    return HTML("<script type='module' src='https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js'></script>"
                "<div style='display:flex;gap:12px'>" + "".join(blocks) + "</div>")

if RUN_INFERENCE:
    display(IPyImage(filename=image, width=256))
    display(glb_viewer([("base", f"{GEN}/base.glb"), ("LoRA", f"{GEN}/lora.glb")]))

## 9. Download the adapters and outputs

In [ ]:
sh(f"cd {PERSIST} && zip -qr {ROOT}/trellis2_lora_outputs.zip results generated eval -x 'results/*/train.log' || true", check=False)
if os.path.exists(f"{ROOT}/trellis2_lora_outputs.zip"):
    from google.colab import files
    files.download(f"{ROOT}/trellis2_lora_outputs.zip")